# Parse Disease Ontology

- ICD10 <-> MeSH
- DO <-> wiki

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

root_dir = Path.home() / 'scratch/projects/befunddolmetscher'

In [ ]:
import sys

sys.path.append((root_dir / 'search').as_posix())

In [ ]:
import requests

file_url = "https://raw.githubusercontent.com/DiseaseOntology/HumanDiseaseOntology/refs/heads/main/src/ontology/HumanDO.json"
human_disease_ontology = requests.get(file_url).json()['graphs'][0]['nodes']

In [ ]:
import json
# need to convert supplemental concept ids -> descriptor ids for MeSH
with (root_dir / 'data/MeSH_supp2023.json').open('r') as f:
    supp_ui = json.load(f)

In [ ]:
from tqdm.auto import tqdm
from collections import defaultdict
from pathlib import Path
import unicodedata

def normalize(s):
    if isinstance(s, str):
        return unicodedata.normalize('NFKD', s).strip()
    return s

records = []
do2wiki = defaultdict(list)
for node in tqdm(human_disease_ontology):
    if 'meta' in node and node['id'].startswith('http://purl.obolibrary.org/obo/DOID_'):
        tag, doid = Path(node['id']).stem.split('_', 1)
        assert tag == 'DOID'
        record = {
            'id': doid,
            'url': node['id'],
            'title': normalize(node.get('lbl')),
            'text': normalize(node['meta']['definition']['val']) if 'definition' in node['meta'] else None,
            'synonyms': '; '.join([normalize(s['val']) for s in node['meta']['synonyms']]) if 'synonyms' in node['meta'] else None
        }
        external_ids = {}
        for key in node['meta'].get('xrefs', []):
            k, v = key['val'].split(':', 1)
            if k == 'MESH' and v.startswith('C') and v in supp_ui:
                v = ';'.join(supp_ui[v]['descriptor_ui'])
            external_ids[k] = normalize(v)
        record.update(external_ids)
        records.append(record)
    
        if 'definition' in node['meta']:
            for url in node['meta']['definition'].get('xrefs', []):
                if url[4:].startswith('http://en.wikipedia.org'):
                    do2wiki[record['id']].append(url[4:])

len(records), len(do2wiki)

In [ ]:
import pandas as pd
wiki = []
for k, v in do2wiki.items():
    for u in set(v):
        wiki.append({'doid': k, 'wikipedia': u})
wiki_df = pd.DataFrame.from_records(wiki)
wiki_df.to_csv(root_dir / 'data/disease_ontology2wikipedia.tsv', sep="\t", index=False)

flatten synonyms

In [ ]:
import pandas as pd
from copy import deepcopy

data_map = {record['id']: record for record in records}
df = pd.DataFrame.from_records(records)
df = df.drop_duplicates(subset=['id', 'title'])
for name, row in df.iterrows():
    if isinstance(row['synonyms'], str) and len(row['synonyms']) > 0:
        for s in row['synonyms'].split(';'):
            if s.strip() != row['title']:
                record = deepcopy(data_map[row['id']])
                record['title'] = s.strip()
                records.append(record)
df = pd.DataFrame.from_records(records)
df = df.drop_duplicates(subset=['id', 'title']) # keep first
df.to_csv(root_dir / 'data/disease_ontology.tsv', sep="\t", index=False)

In [ ]:
headers = df.columns.tolist()
headers

In [ ]:
import pandas as pd
df = pd.read_csv(
    root_dir / 'data/disease_ontology.tsv', sep="\t", index_col=False,
    dtype={k: str for k in headers}
).fillna('')
df.head(5)

In [ ]:
import requests
def embed(contents: list, host: str="localhost", port: str="8080"):
    """embed texts"""
    response = requests.post(
        f"http://{host}:{port}/embedding",
        headers={"Content-Type": "application/json"},
        json={"content": contents},
    )
    return response.json()

In [ ]:
from tqdm.auto import tqdm
import numpy as np
import json
output_path = (root_dir / 'results/disease_ontology_emb.jsonl')

def embed_disease_ontology(df):
    batch_size = 32
    for key, group in tqdm(df.groupby(np.arange(len(df))//batch_size), total=(len(df)//batch_size +1)):
        doids = group.id.to_list()
        titles = group.title.to_list()
        texts = []
        for name, row in group.iterrows():
            texts.append((row['title'] + '\n\n' + row['text']).strip())
        
        assert len(texts) == len(doids) == len(titles)
    
        vectors = embed(contents=texts, host='localhost', port='8181')
        assert len(vectors) == len(texts)
    
        # save in json file
        for v, i, t in zip(vectors, doids, titles):
            with output_path.open('a') as f:
                v['doid'] = i
                v['title'] = t
                f.write(json.dumps(v) + '\n')

In [ ]:
import json

output_path = (root_dir / 'results/disease_ontology_emb.jsonl')
if not output_path.exists():
    embed_disease_ontology(df)

vectors = [json.loads(l) for l in output_path.read_text().splitlines()]
vectors = {(v['doid'], v['title']): v['embedding'][0] for v in vectors if len(v['embedding'][0]) == 4096}
len(vectors)

In [ ]:
from tqdm.auto import tqdm

metadatas = []
embeds = []
for name, row in tqdm(df.iterrows()):
    metadatas.append({'doid': row['id'], 'icd10': row['ICD10CM'].lower(), 'mesh': row['MESH']})
    embeds.append(vectors[(row['id'], row['title'])])

In [ ]:
from sqlitevec import SQLiteVec

disease_ontology_table = SQLiteVec.from_texts(
    titles=df.title.to_list(),
    texts=df.text.to_list(),
    embeds=embeds,
    metadatas=metadatas,
    table=f"disease_ontology",
    db_file="disease_ontology.db",
    lang='en',
)

In [ ]:
found = disease_ontology_table.metadata_search(
    icd10=['k35', 'k36', 'k37', 'k38'] #exact match
)
for f in found: # relevance rank from BM25
    print(f['doid'], f['title'], f['icd10'], f['mesh'])

In [ ]:
from IPython.display import display, HTML

found = disease_ontology_table.fulltext_search_with_highlight(
    query='title: Appendicitis' #lemma match
)
for f in found: # relevance rank from BM25
    print(f['rank'], f['doid'], f['icd10'], f['mesh'])
    display(HTML(f['title']))
    print('---')

In [ ]:
from IPython.display import display, HTML

found = disease_ontology_table.fulltext_search_with_highlight(
    query='title_lower: "parkinson disease"' #case-insensitive match
)
for f in found: # relevance rank from BM25
    print(f['rank'], f['doid'], f['icd10'], f['mesh'])
    display(HTML(f['title']))
    print('---')